### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Install Required Libraries

In [ ]:
!pip install transformers accelerate scikit-learn -qqq

### Define Paths for Models and Data

In [ ]:
import os

# Update these paths with your actual Google Drive paths
TRANSFORMER_MODEL_PATH = "/content/drive/MyDrive/InformationRet/final_sota_model"
MODEL_ARTIFACTS_PATH = "/content/drive/MyDrive/InformationRet"
LABELS_PATH = "/content/drive/MyDrive/InformationRet/distilbert_results"
TEST_CSV = "/content/drive/MyDrive/InformationRet/test.csv"

print("Current directory:", os.getcwd())
print(f"Transformer model path set to: {TRANSFORMER_MODEL_PATH}")
print(f"Other model artifacts path set to: {MODEL_ARTIFACTS_PATH}")
print(f"Labels path set to: {LABELS_PATH}")
print(f"Test CSV path set to: {TEST_CSV}")

# Verify paths exist (optional, but good practice)
if not os.path.exists(TRANSFORMER_MODEL_PATH):
    print(f"Warning: TRANSFORMER_MODEL_PATH does not exist: {TRANSFORMER_MODEL_PATH}")
if not os.path.exists(MODEL_ARTIFACTS_PATH):
    print(f"Warning: MODEL_ARTIFACTS_PATH does not exist: {MODEL_ARTIFACTS_PATH}")
if not os.path.exists(LABELS_PATH):
    print(f"Warning: LABELS_PATH does not exist: {LABELS_PATH}")
if not os.path.exists(TEST_CSV):
    print(f"Warning: TEST_CSV does not exist: {TEST_CSV}")

### Load Data and Models

In [ ]:
import time
import torch
import joblib
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm # Import tqdm for progress bars

df = pd.read_csv(TEST_CSV)
print("Columns in dataset:", df.columns.tolist())

TEXT_COL = "text"
LABEL_COL = "category"

texts = df[TEXT_COL].fillna("").tolist()
true_labels_raw = df[LABEL_COL].tolist()

labels_data = joblib.load(os.path.join(LABELS_PATH, "labels.pkl"))
label2id = labels_data["label2id"]
id2label = {v: k for k, v in label2id.items()}

filtered_texts = []
true_labels = []

for text, label in zip(texts, true_labels_raw):
    if label in label2id:
        filtered_texts.append(text)
        true_labels.append(label2id[label])

texts = filtered_texts
true_labels = [int(x) for x in true_labels]

print(f"Using {len(texts)} valid samples")

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_PATH, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    TRANSFORMER_MODEL_PATH,
    local_files_only=True,
    use_safetensors=True  # Explicitly use your .safetensors file
)

# Automatically use GPU if available, otherwise fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

nb_model = joblib.load(os.path.join(MODEL_ARTIFACTS_PATH, "nb_model.pkl"))
lr_model = joblib.load(os.path.join(MODEL_ARTIFACTS_PATH, "lr_model.pkl"))

print("All models loaded successfully")
print(f"Using device: {device}")

### Define Prediction Functions

In [ ]:
from tqdm.auto import tqdm

def predict_transformer(texts, batch_size=32):
    preds = []
    # Use tqdm for a visual progress bar
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting Transformer"):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy().tolist()

        preds.extend(batch_preds)

    return preds

def normalize_preds(raw_preds):
    preds = []
    for p in raw_preds:
        if isinstance(p, str):
            p = p.strip()
            preds.append(label2id[p] if p in label2id else 0)
        else:
            preds.append(int(p))
    return preds

### Generate Predictions

In [ ]:
print("Generating Transformer predictions...")
start = time.time()
preds_tf = predict_transformer(texts, batch_size=1024)
print(f"Transformer done in {time.time() - start:.2f} seconds")

print("Generating Naive Bayes predictions...")
start = time.time()
preds_nb = normalize_preds(nb_model.predict(texts))
print(f"Naive Bayes done in {time.time() - start:.2f} seconds")

print("Generating Logistic Regression predictions...")
start = time.time()
preds_lr = normalize_preds(lr_model.predict(texts))
print(f"Logistic Regression done in {time.time() - start:.2f} seconds")

### Evaluate Models and Plot Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

def plot_cm(true, pred, title, filename):
    labels = sorted(label2id.values()) # Use actual integer labels for CM calculation
    cm = confusion_matrix(true, pred, labels=labels)

    # Ensure display labels are correctly ordered and mapped to names
    display_labels_names = [id2label[i] for i in labels]

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=display_labels_names
    )
    fig, ax = plt.subplots(figsize=(14, 12))
    disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False, values_format="d", text_kw={"fontsize": 6})
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")

os.makedirs("results", exist_ok=True)

plot_cm(true_labels, preds_tf, "Transformer Confusion Matrix", "results/transformer_cm.png")
plot_cm(true_labels, preds_nb, "Naive Bayes Confusion Matrix", "results/nb_cm.png")
plot_cm(true_labels, preds_lr, "Logistic Regression Confusion Matrix", "results/lr_cm.png")

print("Saved confusion matrices to /results folder")

# Generate and print classification reports for each model
print("\n--- Classification Report: Transformer ---")
print(classification_report(true_labels, preds_tf, target_names=[id2label[i] for i in sorted(id2label.keys())]))

print("\n--- Classification Report: Naive Bayes ---")
print(classification_report(true_labels, preds_nb, target_names=[id2label[i] for i in sorted(id2label.keys())]))

print("\n--- Classification Report: Logistic Regression ---")
print(classification_report(true_labels, preds_lr, target_names=[id2label[i] for i in sorted(id2label.keys())]))

### Top Words Per Category

In [ ]:
import numpy as np

# Assuming nb_model and lr_model are scikit-learn Pipeline objects
# Extract the vectorizer and classifier from the Naive Bayes pipeline
vectorizer_nb = None
classifier_nb = None
for name, step in nb_model.steps:
    if hasattr(step, 'get_feature_names_out'): # Check for vectorizer methods
        vectorizer_nb = step
    elif hasattr(step, 'feature_log_prob_'): # Check for Naive Bayes classifier attribute
        classifier_nb = step

# Extract the vectorizer and classifier from the Logistic Regression pipeline
vectorizer_lr = None
classifier_lr = None
for name, step in lr_model.steps:
    if hasattr(step, 'get_feature_names_out'): # Check for vectorizer methods
        vectorizer_lr = step
    elif hasattr(step, 'coef_'): # Check for Logistic Regression classifier attribute
        classifier_lr = step

if vectorizer_nb is None or classifier_nb is None:
    print("Error: Could not extract vectorizer or classifier from Naive Bayes pipeline.")
else:
    feature_names = vectorizer_nb.get_feature_names_out()
    print("\n--- Top 10 words for each category (Naive Bayes) ---")
    for i, category_name in id2label.items():
        if i < classifier_nb.feature_log_prob_.shape[0]: # Ensure class index exists
            log_probs = classifier_nb.feature_log_prob_[i]
            top_features_idx = log_probs.argsort()[-10:][::-1] # Get indices of top 10 log probabilities
            top_words = [feature_names[idx] for idx in top_features_idx]
            print(f"Category '{category_name}': {', '.join(top_words)}")
        else:
            print(f"Warning: Category index {i} not found in Naive Bayes model.")

if vectorizer_lr is None or classifier_lr is None:
    print("Error: Could not extract vectorizer or classifier from Logistic Regression pipeline.")
else:
    feature_names = vectorizer_lr.get_feature_names_out()
    print("\n--- Top 10 words for each category (Logistic Regression) ---")
    for i, category_name in id2label.items():
        if i < classifier_lr.coef_.shape[0]: # Ensure class index exists
            coefs = classifier_lr.coef_[i]
            top_features_idx = coefs.argsort()[-10:][::-1] # Get indices of top 10 positive coefficients
            top_words = [feature_names[idx] for idx in top_features_idx]
            print(f"Category '{category_name}': {', '.join(top_words)}")
        else:
            print(f"Warning: Category index {i} not found in Logistic Regression model.")